In [1]:
#!/usr/bin/env python3
"""
H2 VQE with UCCSD + TOPAZ (5 UPTE layers) + MMA + L2 regularization
"""

import numpy as np
import scipy.linalg as la

from qiskit.quantum_info import SparsePauliOp, Operator
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD


# =====================================================
# GLOBAL CONFIG
# =====================================================

N_LAYERS_TOPAZ = 5
PAULI_STRINGS = ["XXII", "YYII", "ZZII", "IIZZ", "ZZII"]
LAMBDA_REG_PHASE1 = 0
LAMBDA_REG_PHASE2 = 0

DEPOL_BASE = 0.005
DEPOL_CORR_LENGTH = 2.0
DEPOL_CORR_STRENGTH = 0.3

AMP_DAMP_BASE = 0.01
AMP_DAMP_CORR_LENGTH = 2.0
AMP_DAMP_CORR_STRENGTH = 0.25

PHASE_DAMP_BASE = 0.005
PHASE_DAMP_CORR_LENGTH = 2.0
PHASE_DAMP_CORR_STRENGTH = 0.3

np.random.seed(42)

# =====================================================
# BASIC MATRICES
# =====================================================

I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def build_full_operator_from_list(oplist):
    if not oplist:
        return np.eye(1, dtype=complex)
    fullop = oplist[0]
    for qidx in range(1, len(oplist)):
        fullop = np.kron(fullop, oplist[qidx])
    return fullop

def pauli_string_to_matrix(pstr, n_qubits):
    op = SparsePauliOp(pstr, coeffs=[1.0])
    return op.to_matrix()

# =====================================================
# NOISE CHANNELS (CORRELATED)
# =====================================================

def apply_correlated_depolarizing_noise(rho_ideal, n_qubits):
    rho_current = rho_ideal.copy()
    for qubit in range(n_qubits):
        p_single = DEPOL_BASE
        K_ops = [
            np.sqrt(1 - 3*p_single/4) * I2,
            np.sqrt(p_single/4) * X,
            np.sqrt(p_single/4) * Y,
            np.sqrt(p_single/4) * Z,
        ]
        rho_q_noisy = np.zeros_like(rho_current, dtype=complex)
        for K in K_ops:
            oplist = [I2]*n_qubits
            oplist[qubit] = K
            K_full = build_full_operator_from_list(oplist)
            rho_q_noisy += K_full @ rho_current @ K_full.conj().T
        rho_current = rho_q_noisy

    for q1 in range(n_qubits - 1):
        q2 = q1 + 1
        distance = 1.0
        p_corr = DEPOL_CORR_STRENGTH * DEPOL_BASE * np.exp(-distance / DEPOL_CORR_LENGTH)
        if p_corr < 1e-12:
            continue
        rho_2q_noisy = (1.0 - p_corr) * rho_current
        for P1, P2 in [(X, X), (Y, Y), (Z, Z)]:
            oplist = [I2]*n_qubits
            oplist[q1] = P1
            oplist[q2] = P2
            P_full = build_full_operator_from_list(oplist)
            rho_2q_noisy += (p_corr/3.0) * (P_full @ rho_current @ P_full.conj().T)
        rho_current = rho_2q_noisy

    tr = np.trace(rho_current)
    if abs(tr) > 1e-12:
        rho_current /= tr
    return rho_current

def apply_correlated_amplitude_damping(rho_ideal, n_qubits):
    rho_current = rho_ideal.copy()
    gamma_base = AMP_DAMP_BASE
    K0 = np.array([[1, 0], [0, np.sqrt(1 - gamma_base)]], dtype=complex)
    K1 = np.array([[0, np.sqrt(gamma_base)], [0, 0]], dtype=complex)
    K_ops = [K0, K1]

    for qubit in range(n_qubits):
        rho_q_noisy = np.zeros_like(rho_current, dtype=complex)
        for K in K_ops:
            oplist = [I2]*n_qubits
            oplist[qubit] = K
            K_full = build_full_operator_from_list(oplist)
            rho_q_noisy += K_full @ rho_current @ K_full.conj().T
        rho_current = rho_q_noisy

    for q1 in range(n_qubits - 1):
        q2 = q1 + 1
        distance = 1.0
        p_corr = AMP_DAMP_CORR_STRENGTH * AMP_DAMP_BASE * np.exp(-distance / AMP_DAMP_CORR_LENGTH)
        if p_corr < 1e-12:
            continue
        rho_2q_noisy = (1.0 - p_corr) * rho_current
        for K1a, K1b in [(K0, K0), (K0, K1), (K1, K0), (K1, K1)]:
            oplist = [I2]*n_qubits
            oplist[q1] = K1a
            oplist[q2] = K1b
            K_full = build_full_operator_from_list(oplist)
            rho_2q_noisy += (p_corr/4.0) * (K_full @ rho_current @ K_full.conj().T)
        rho_current = rho_2q_noisy

    tr = np.trace(rho_current)
    if abs(tr) > 1e-12:
        rho_current /= tr
    return rho_current

def apply_correlated_phase_damping(rho_ideal, n_qubits):
    rho_current = rho_ideal.copy()
    lambda_base = PHASE_DAMP_BASE
    K0 = np.array([[1, 0], [0, np.sqrt(1 - lambda_base)]], dtype=complex)
    K1 = np.array([[0, 0], [0, np.sqrt(lambda_base)]], dtype=complex)
    K_ops = [K0, K1]

    for qubit in range(n_qubits):
        rho_q_noisy = np.zeros_like(rho_current, dtype=complex)
        for K in K_ops:
            oplist = [I2]*n_qubits
            oplist[qubit] = K
            K_full = build_full_operator_from_list(oplist)
            rho_q_noisy += K_full @ rho_current @ K_full.conj().T
        rho_current = rho_q_noisy

    for q1 in range(n_qubits - 1):
        q2 = q1 + 1
        distance = 1.0
        p_corr = PHASE_DAMP_CORR_STRENGTH * PHASE_DAMP_BASE * np.exp(-distance / PHASE_DAMP_CORR_LENGTH)
        if p_corr < 1e-12:
            continue
        rho_2q_noisy = (1.0 - p_corr) * rho_current
        for K1a, K1b in [(K0, K0), (K0, K1), (K1, K0), (K1, K1)]:
            oplist = [I2]*n_qubits
            oplist[q1] = K1a
            oplist[q2] = K1b
            K_full = build_full_operator_from_list(oplist)
            rho_2q_noisy += (p_corr/4.0) * (K_full @ rho_current @ K_full.conj().T)
        rho_current = rho_2q_noisy

    tr = np.trace(rho_current)
    if abs(tr) > 1e-12:
        rho_current /= tr
    return rho_current

def apply_all_correlated_noises(rho_ideal, n_qubits):
    rho = apply_correlated_depolarizing_noise(rho_ideal, n_qubits)
    rho = apply_correlated_amplitude_damping(rho, n_qubits)
    rho = apply_correlated_phase_damping(rho, n_qubits)
    return rho

# =====================================================
# MMA OPTIMIZER
# =====================================================

class MMAOptimizer:
    def __init__(self, nparams):
        self.nparams = nparams
        self.movelimit = 1.0
        self.gamma = 0.9
        self.L = None
        self.U = None
        self.previousparams = None
        self.previousloss = None

    def initialize_bounds(self, initialparams):
        self.L = initialparams - 1.0
        self.U = initialparams + 1.0
        self.previousparams = initialparams.copy()
        self.previousloss = None

    def solve_convex_subproblem(self, currentparams, grad):
        xnew = np.zeros_like(currentparams)
        for i in range(self.nparams):
            gi = grad[i]
            xi = currentparams[i]
            Li = self.L[i]
            Ui = self.U[i]

            if gi > 0:
                pi = abs(gi) / max(Ui - xi, 1e-12)
                qi = 0.0
            else:
                pi = 0.0
                qi = abs(gi) / max(xi - Li, 1e-12)

            denominator = pi*(Ui - xi + 1e-12) + qi*(xi - Li + 1e-12)
            if denominator < 1e-12:
                xnew[i] = xi
            else:
                numerator = pi*(Ui - xi + 1e-12) - qi*(xi - Li + 1e-12)
                xnew[i] = xi - numerator/denominator

            xnew[i] = max(xi - self.movelimit, min(xi + self.movelimit, xnew[i]))
            xnew[i] = np.clip(xnew[i], Li + 1e-6, Ui - 1e-6)
        return xnew

    def update_asymptotes(self, currentparams, currentloss):
        progressgood = True
        if self.previousloss is not None:
            progressgood = (currentloss < self.previousloss - 1e-8)
        for i in range(self.nparams):
            if progressgood:
                self.L[i] = currentparams[i] - 1.5*self.gamma*(currentparams[i] - self.L[i])
                self.U[i] = currentparams[i] + 1.5*self.gamma*(self.U[i] - currentparams[i])
            else:
                self.L[i] = currentparams[i] - 0.7*self.gamma*(currentparams[i] - self.L[i])
                self.U[i] = currentparams[i] + 0.7*self.gamma*(self.U[i] - currentparams[i])
        self.L = np.minimum(self.L, self.U - 1e-4)
        self.U = np.maximum(self.U, self.L + 1e-4)
        self.previousparams = currentparams.copy()
        self.previousloss = currentloss

def finite_diff_gradient(params, objective_fn, eps_default=2e-3):
    n = len(params)
    grad = np.zeros_like(params)
    for i in range(n):
        eps = eps_default
        p_plus = params.copy()
        p_minus = params.copy()
        p_plus[i] += eps
        p_minus[i] -= eps
        f_plus = objective_fn(p_plus)
        f_minus = objective_fn(p_minus)
        grad[i] = (f_plus - f_minus)/(2*eps)
    return grad

def run_mma_optimization(initialparams, objective_fn, maxiterations=120, label=""):
    mma = MMAOptimizer(len(initialparams))
    mma.initialize_bounds(initialparams)
    currentparams = initialparams.copy()
    currentloss = objective_fn(currentparams)

    bestparams = currentparams.copy()
    bestloss = currentloss
    stagnation = 0

    print(f"    Initial {label} loss: {currentloss:.6f}")

    for it in range(maxiterations):
        grad = finite_diff_gradient(currentparams, objective_fn, eps_default=2e-3)
        newparams = mma.solve_convex_subproblem(currentparams, grad)
        newloss = objective_fn(newparams)

        if newloss < currentloss - 1e-5:
            currentparams = newparams
            currentloss = newloss
            stagnation = 0
        else:
            stagnation += 1

        if currentloss < bestloss:
            bestloss = currentloss
            bestparams = currentparams.copy()

        mma.update_asymptotes(currentparams, currentloss)

        if (it+1) % 10 == 0 or it == 0:
            print(f"    Iter {it+1:3d}: loss = {currentloss:.6f}")

        if stagnation >= 40:
            print(f"    Stopping MMA ({label}): stagnation at iter {it+1}")
            break

    print(f"    MMA ({label}) complete. Best loss = {bestloss:.6f}")
    return bestparams, bestloss

# =====================================================
# H2 PROBLEM (4-QUBIT, JW)
# =====================================================

print("="*70)
print(" H2 VQE + UCCSD + TOPAZ (5 layers) + MMA + L2 regularization")
print("="*70)

driver = PySCFDriver(atom='H 0 0 0; H 0 0 0.735', basis='sto3g', unit=DistanceUnit.ANGSTROM)
es_problem = driver.run()
nuclear_repulsion = es_problem.nuclear_repulsion_energy

mapper = JordanWignerMapper()
H2_op = mapper.map(es_problem.hamiltonian.second_q_op())

H_electronic = H2_op.to_matrix()
n_qubits = H2_op.num_qubits

# Add nuclear repulsion to get total Hamiltonian
H_matrix = H_electronic + nuclear_repulsion * np.eye(2**n_qubits, dtype=complex)

print(f"  n_qubits (JW, no taper): {n_qubits}")
print(f"  Nuclear repulsion:       {nuclear_repulsion:.6f} Ha")

# FCI reference (kernel returns TOTAL energy directly)
from pyscf import gto, fci
mol = gto.Mole()
mol.atom = 'H 0 0 0; H 0 0 0.735'
mol.basis = 'sto3g'
mol.build()
mf = mol.RHF().run()
fci_energy_total = fci.FCI(mf).kernel()[0]
print(f"  FCI total energy:        {fci_energy_total:.6f} Ha")

# Setup HF and UCCSD
num_spatial_orbitals = es_problem.num_spatial_orbitals
num_particles = es_problem.num_particles
hf = HartreeFock(num_spatial_orbitals, num_particles, mapper)
uccsd = UCCSD(num_spatial_orbitals, num_particles, mapper, initial_state=hf)
n_theta = uccsd.num_parameters

print(f"  UCCSD parameters: {n_theta}")

def circuit_to_unitary(circuit):
    return Operator(circuit).data

# Get HF unitary to use as starting point
U_hf = circuit_to_unitary(hf)

# =====================================================
# TOPAZ / UPTE (5 layers) ON 4 QUBITS
# =====================================================

N_OPS = len(PAULI_STRINGS)
PAULI_MATS = [pauli_string_to_matrix(p, n_qubits) for p in PAULI_STRINGS]

def build_topaz_from_rho_tau(rho_all, tau_all):
    dim = 2**n_qubits
    U = np.eye(dim, dtype=complex)
    for layer in range(N_LAYERS_TOPAZ):
        rho = rho_all[layer]
        tau = tau_all[layer]
        B = np.zeros((dim, dim), dtype=complex)
        for j, H_j in enumerate(PAULI_MATS):
            B += rho[j] * la.expm(-1j * H_j * tau[j])
        U_layer, _ = la.polar(B)
        det = np.linalg.det(U_layer)
        if abs(det) > 1e-12:
            U_layer *= det ** (-1.0 / dim)
        U = U_layer @ U
    return U

def unpack_rho_tau(params_struct):
    n_rho = N_LAYERS_TOPAZ * N_OPS
    n_tau = N_LAYERS_TOPAZ * N_OPS
    rho_flat = params_struct[:n_rho]
    tau_flat = params_struct[n_rho:n_rho + n_tau]
    rho_layers = []
    tau_layers = []
    for i in range(N_LAYERS_TOPAZ):
        rho_layer = np.abs(rho_flat[i*N_OPS:(i+1)*N_OPS])
        rho_layer /= (rho_layer.sum() + 1e-12)
        tau_layer = np.clip(tau_flat[i*N_OPS:(i+1)*N_OPS], 0.01, np.pi/2)
        rho_layers.append(rho_layer)
        tau_layers.append(tau_layer)
    return rho_layers, tau_layers

# =====================================================
# PHASE 1: MMA ON TOPAZ ONLY (ρ, τ) + L2
# =====================================================

print("\n[1] Phase 1: MMA on TOPAZ (ρ, τ) only + L2")

def phase1_objective(params_struct):
    rho_all, tau_all = unpack_rho_tau(params_struct)
    U_topaz = build_topaz_from_rho_tau(rho_all, tau_all)

    # FIX: Start from HF state, not |0000⟩
    U_total = U_topaz @ U_hf

    psi0 = np.zeros(2**n_qubits, dtype=complex)
    psi0[0] = 1.0
    psi_ideal = U_total @ psi0
    rho_ideal = np.outer(psi_ideal, psi_ideal.conj())
    rho_noisy = apply_all_correlated_noises(rho_ideal, n_qubits)

    energy = np.real(np.trace(rho_noisy @ H_matrix))
    reg = LAMBDA_REG_PHASE1 * np.sum(params_struct**2)
    return energy + reg

n_rho = N_LAYERS_TOPAZ * N_OPS
n_tau = N_LAYERS_TOPAZ * N_OPS
n_struct = n_rho + n_tau

rho_init_layers = np.random.dirichlet(np.ones(N_OPS)*2, size=N_LAYERS_TOPAZ)
tau_init_layers = np.random.uniform(0.1, np.pi/3, size=(N_LAYERS_TOPAZ, N_OPS))
params_struct_init = np.concatenate([rho_init_layers.flatten(), tau_init_layers.flatten()])

params_struct_opt, best_e_phase1_reg = run_mma_optimization(
    params_struct_init,
    phase1_objective,
    maxiterations=120,
    label="Phase 1 (ρ,τ) + L2"
)

rho_opt, tau_opt = unpack_rho_tau(params_struct_opt)
U_topaz_opt = build_topaz_from_rho_tau(rho_opt, tau_opt)

def phase1_energy_only(params_struct):
    rho_all, tau_all = unpack_rho_tau(params_struct)
    U_topaz = build_topaz_from_rho_tau(rho_all, tau_all)
    U_total = U_topaz @ U_hf
    psi0 = np.zeros(2**n_qubits, dtype=complex)
    psi0[0] = 1.0
    psi_ideal = U_total @ psi0
    rho_ideal = np.outer(psi_ideal, psi_ideal.conj())
    rho_noisy = apply_all_correlated_noises(rho_ideal, n_qubits)
    return np.real(np.trace(rho_noisy @ H_matrix))

best_e_phase1 = phase1_energy_only(params_struct_opt)
print(f"    Best Phase 1 total energy (no reg term): {best_e_phase1:.6f} Ha")

# =====================================================
# PHASE 2: Δρ, Δτ + UCCSD θ + L2
# =====================================================

print("\n[2] Phase 2: local Δ(ρ, τ) + UCCSD θ via MMA + L2")

params_struct_opt_flat = params_struct_opt.copy()
n_struct_params = len(params_struct_opt_flat)
STRUCT_TRUST = 0.1

def phase2_objective(params_all):
    delta_struct = params_all[:n_struct_params]
    theta = params_all[n_struct_params:]

    delta_struct = np.clip(delta_struct, -STRUCT_TRUST, STRUCT_TRUST)
    struct_refined = params_struct_opt_flat * (1.0 + delta_struct)

    rho_ref, tau_ref = unpack_rho_tau(struct_refined)
    U_topaz_ref = build_topaz_from_rho_tau(rho_ref, tau_ref)

    # UCCSD already includes HF as initial_state
    bound_ucc = uccsd.assign_parameters(theta)
    U_ucc = circuit_to_unitary(bound_ucc)

    # Apply TOPAZ after UCCSD (which includes HF)
    U_total = U_topaz_ref @ U_ucc

    psi0 = np.zeros(2**n_qubits, dtype=complex)
    psi0[0] = 1.0
    psi_ideal = U_total @ psi0
    rho_ideal = np.outer(psi_ideal, psi_ideal.conj())
    rho_noisy = apply_all_correlated_noises(rho_ideal, n_qubits)

    energy = np.real(np.trace(rho_noisy @ H_matrix))
    reg = LAMBDA_REG_PHASE2 * np.sum(params_all**2)
    return energy + reg

delta_struct_init = np.zeros(n_struct_params)
theta_init = np.zeros(n_theta)
params_phase2_init = np.concatenate([delta_struct_init, theta_init])

params_phase2_opt, best_e_phase2_reg = run_mma_optimization(
    params_phase2_init,
    phase2_objective,
    maxiterations=120,
    label="Phase 2 (Δstruct + θ_UCCSD) + L2"
)

delta_struct_opt = np.clip(params_phase2_opt[:n_struct_params], -STRUCT_TRUST, STRUCT_TRUST)
theta_opt = params_phase2_opt[n_struct_params:]
struct_final = params_struct_opt_flat * (1.0 + delta_struct_opt)
rho_final, tau_final = unpack_rho_tau(struct_final)

def phase2_energy_only(params_all):
    delta_struct = params_all[:n_struct_params]
    theta = params_all[n_struct_params:]
    delta_struct = np.clip(delta_struct, -STRUCT_TRUST, STRUCT_TRUST)
    struct_refined = params_struct_opt_flat * (1.0 + delta_struct)
    rho_ref, tau_ref = unpack_rho_tau(struct_refined)
    U_topaz_ref = build_topaz_from_rho_tau(rho_ref, tau_ref)
    bound_ucc = uccsd.assign_parameters(theta)
    U_ucc = circuit_to_unitary(bound_ucc)
    U_total = U_topaz_ref @ U_ucc
    psi0 = np.zeros(2**n_qubits, dtype=complex)
    psi0[0] = 1.0
    psi_ideal = U_total @ psi0
    rho_ideal = np.outer(psi_ideal, psi_ideal.conj())
    rho_noisy = apply_all_correlated_noises(rho_ideal, n_qubits)
    return np.real(np.trace(rho_noisy @ H_matrix))

best_e_phase2 = phase2_energy_only(params_phase2_opt)
print(f"    Best Phase 2 total energy (no reg term): {best_e_phase2:.6f} Ha")

# =====================================================
# FINAL REPORT
# =====================================================

print("\n" + "="*70)
print(" FINAL RESULTS")
print("="*70)
print(f"  FCI total (reference):               {fci_energy_total:.6f} Ha")
print(f"  TOPAZ-only total (Phase 1):          {best_e_phase1:.6f} Ha")
print(f"  TOPAZ+UCCSD (Phase 2) total:         {best_e_phase2:.6f} Ha")
print(f"  Error vs FCI (Phase 2):              {(best_e_phase2 - fci_energy_total)*1000:.2f} mHa")
print(f"  λ_phase1 = {LAMBDA_REG_PHASE1:.1e}, λ_phase2 = {LAMBDA_REG_PHASE2:.1e}")
print("="*70)

 H2 VQE + UCCSD + TOPAZ (5 layers) + MMA + L2 regularization
  n_qubits (JW, no taper): 4
  Nuclear repulsion:       0.719969 Ha
converged SCF energy = -1.116998996754
  FCI total energy:        -1.137306 Ha
  UCCSD parameters: 3

[1] Phase 1: MMA on TOPAZ (ρ, τ) only + L2
    Initial Phase 1 (ρ,τ) + L2 loss: -0.752742
    Iter   1: loss = -1.097708
    Iter  10: loss = -1.097834
    Iter  20: loss = -1.097834
    Iter  30: loss = -1.097834
    Iter  40: loss = -1.097834
    Stopping MMA (Phase 1 (ρ,τ) + L2): stagnation at iter 42
    MMA (Phase 1 (ρ,τ) + L2) complete. Best loss = -1.097834
    Best Phase 1 total energy (no reg term): -1.097834 Ha

[2] Phase 2: local Δ(ρ, τ) + UCCSD θ via MMA + L2
    Initial Phase 2 (Δstruct + θ_UCCSD) + L2 loss: -1.097834


/opt/miniconda3/envs/qiskit_gaara/lib/python3.14/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/opt/miniconda3/envs/qiskit_gaara/lib/python3.14/site-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


    Iter   1: loss = -1.097834
    Iter  10: loss = -1.114066
    Iter  20: loss = -1.116423
    Iter  30: loss = -1.116506
    Iter  40: loss = -1.116506
    Iter  50: loss = -1.116506
    Iter  60: loss = -1.116506
    Stopping MMA (Phase 2 (Δstruct + θ_UCCSD) + L2): stagnation at iter 65
    MMA (Phase 2 (Δstruct + θ_UCCSD) + L2) complete. Best loss = -1.116506
    Best Phase 2 total energy (no reg term): -1.116506 Ha

 FINAL RESULTS
  FCI total (reference):               -1.137306 Ha
  TOPAZ-only total (Phase 1):          -1.097834 Ha
  TOPAZ+UCCSD (Phase 2) total:         -1.116506 Ha
  Error vs FCI (Phase 2):              20.80 mHa
  λ_phase1 = 0.0e+00, λ_phase2 = 0.0e+00
